You can download books in .txt format from the [Gutenberg Project](https://www.gutenberg.org/browse/scores/top).

In [ ]:
!pip install -q chromadb openai

In [ ]:
from pathlib import Path
from pprint import pprint

def print_response(response):
    print(f"Reponse id: {response.id}")
    print(f"Status: {response.status}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached) | Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'=' * 20} [Text] {'=' * 20}")
    print(response.output_text)

In [ ]:
from chromadb import PersistentClient

PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)
chroma_collection = chroma_client.get_or_create_collection(name="books")

In [ ]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=api_key)

In [ ]:
file_paths = [Path("/content/Metamorphosis.txt"), Path("/content/Moby Dick.txt"), Path("/content/Alice's Adventures in Wonderland.txt")]

## Index the books

In [ ]:
def index_file(path_to_file):
    with path_to_file.open() as file:
        lines = file.readlines()

    lines = [{ "text": line.strip(), "row": i + 1 } for i, line in enumerate(lines)]
    lines = [line for line in lines if line["text"] != '']

    line_idx = 0
    chunk_length = 6
    chunk_overlap = 2
    chunks = []

    while line_idx < len(lines):
        take = min(chunk_length, len(lines) - line_idx)
        start = line_idx

        chunks.append({
            "text": ' '.join(lines[start + i]["text"] for i in range(take)),
            "row_range": { "from": lines[start]["row"], "to": lines[start + take - 1]["row"] }
        })

        line_idx = line_idx + chunk_length - chunk_overlap


    chroma_collection.add(
        ids=[f"{path_to_file.stem}_{chunk["row_range"]["from"]}_{chunk["row_range"]["to"]}" for chunk in chunks],
        metadatas=[{ "book_name": ..., "ref_start": chunk["row_range"]["from"], "ref_end": chunk["row_range"]["to"] } for chunk in chunks],
        documents=[chunk["text"] for chunk in chunks]
    )


In [ ]:
from rich.console import Console
from rich.table import Table

def print_query_results(result):
    console = Console()

    queries_count = len(result["ids"])
    for i in range(queries_count):
        table = Table(show_lines=True, expand=True)
        table.add_column("#")
        table.add_column("Text")

        results_count = len(result["ids"][i])
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["documents"][i][j])

        console.print(table)

In [ ]:
for i in range(len(file_paths)):
    index_file(file_paths[i])

In [ ]:
queries = ["The nature is beautiful", "In my room is dark", "It is amazing how a person can change"]

In [ ]:
semantic_search_results = chroma_collection.query(
    query_texts=queries
)

In [ ]:
print_query_results(semantic_search_results)

## RAG

In [ ]:
from pydantic import BaseModel, Field

class QuoteLookupRequest(BaseModel):
    """
    Searches the quotes catalog using semantic similarity to find phrases
    matching the user's intent. Use this tool when a user asks about
    the content of a book, a quote reference, or recommendations.
    """
    query: str = Field(description="A natural language search query describing the desired quote (e.g. 'leadership as the most important characterstic of the modern man', 'love will save the world', 'motivation is all you need')")

In [ ]:
quote_lookup_request_schema = QuoteLookupRequest.model_json_schema()

tools = [
    {
        "type": "function",
        "name": "lookup_quote",
        "description": quote_lookup_request_schema["description"],
        "parameters": {
          "type": "object",
          "properties": quote_lookup_request_schema["properties"],
          "required": [*quote_lookup_request_schema["properties"].keys()],
          "additionalProperties": False
        },
        "strict": True
    }
]

In [ ]:
def lookup_quote(req: QuoteLookupRequest):
    query_result = chroma_collection.query(query_texts=[req.query], include=["documents"])

    result_documents_count = len(query_result["ids"][0])

    # TODO: split id in book name and row
    return [{ "id": query_result["ids"][0][i], "text": query_result["documents"][0][i] } for i in range(result_documents_count)]


tool_handlers = {
    "lookup_quote": lambda args: lookup_quote(QuoteLookupRequest(**args))
}

In [ ]:
import json

def interact_with_ai(input, max_iterations_count = 10):
    full_conversation = [*input]

    for i in range(max_iterations_count):
        print(f"Starting iteration #{i + 1}")

        current_response = openai_client.responses.create(
            model="gpt-5-nano",
            input=full_conversation,
            tools=tools
        )

        # TODO: sanitize
        full_conversation.extend(current_response.output)

        tool_calls = [item for item in current_response.output if item.type == "function_call"]
        if len(tool_calls) == 0:
            return full_conversation

        print(f"Found {len(tool_calls)} tool calls.")

        for tc in tool_calls:
            print(tc.name, tc.arguments)
            handler = tool_handlers[tc.name]
            result = handler(json.loads(tc.arguments))

            full_conversation.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": json.dumps(result)
            })

            print(full_conversation[-1])


    raise Exception(f"The AI interaction couldn't finish in {max_iterations_count} iterations.")

In [ ]:
system_prompt = "You are an expert in book recommendations and quote finding. The user will ask you to lookup quotes on a given topic. Always use the \"lookup_query\" tool to find interesting references. The final response should be based solely on the \"lookup_query\" tool call output you get. Do not invent or refer to any quotes that are not part of the \"lookup_query\" output."
user_prompts = [
    "I need to find a quote about the transformational power of love and beauty.",
    "Find a picturesque quote about childhood, dreaming and living effortlessly.",
    "Find quotes about the meaning of life and combine them with other quotes about simple existential obstacles to make the reader feel more motivated.",
    "Find the most dark soul breaking quotes in existence."
]

In [ ]:
for user_prompt in user_prompts:
    conversation = interact_with_ai(
        input=[
            { "role": "developer", "content": system_prompt },
            { "role": "user", "content": user_prompt }
        ]
    )

    print("\n".join(item.text for item in conversation[-1].content))